# pyena ⇄ rENA: numerical parity benchmark

This notebook reproduces, from scratch, the bit-for-bit validation of pyena
against the reference R package
[`rENA`](https://cran.r-project.org/package=rENA) 0.3.1.

Three stages are checked on the same 90-unit toy dataset:

| Stage              | Cells / values | Max absolute difference     |
|--------------------|----------------|------------------------------|
| Adjacency vectors  | 900            | 0 (integer arithmetic)       |
| SVD coordinates    | 180            | < 10⁻¹⁵ (machine ε)          |
| Means rotation     | 180            | < 10⁻¹⁵ (machine ε)          |

## Prerequisites

This notebook requires:

- R installed and accessible from Python
- `rENA` 0.3.1 installed in R: `install.packages("rENA")`
- `rpy2` 3.5+ installed in Python: `pip install rpy2`

If these are not available, the first code cell below will tell you what's
missing. The rest of the pyena test suite (101 tests in `tests/`) does *not*
require rpy2 and runs in pure Python.

In [3]:
import sys

try:
    import rpy2
    import rpy2.robjects as ro
    from rpy2.robjects.packages import importr
    
    try:
        from importlib.metadata import version as _pkg_version
        rpy2_version = _pkg_version("rpy2")
    except Exception:
        rpy2_version = "(unknown)"
    print(f"✓ rpy2 {rpy2_version}")
except ImportError:
    print("✗ rpy2 not found. Install with:  pip install rpy2")
    sys.exit()

r_version = str(ro.r('R.version.string')[0])
print(f"✓ {r_version}")

try:
    rena = importr('rENA')
    rena_version = str(ro.r('as.character(packageVersion("rENA"))')[0])
    print(f"✓ rENA {rena_version}")
    if rena_version != "0.3.1":
        print(f"  Note: this benchmark was developed against rENA 0.3.1. "
              f"Other versions may differ.")
except Exception:
    print(f"✗ rENA not installed in R. From an R console run:")
    print(f"    install.packages(\"rENA\")")
    sys.exit()

print()
print("Environment ready.")

✓ rpy2 3.6.7
✓ R version 4.5.2 (2025-10-31)
✓ rENA 0.3.1

Environment ready.


## Stage 1: Adjacency vectors

The first stage of any ENA pipeline is the moving-stanza adjacency
accumulation. We run `pyena.compute_all_avs` and `rENA::ena.accumulate.data`
on the same input and compare element-wise.

In [4]:
import numpy as np
import pandas as pd
from pyena.core.adjacency import compute_all_avs

# Load the toy dataset (1800 utterances, 90 units, 5 binary codes)
df = pd.read_csv("../tests/data/toy_data_n90.csv")
codes = ["Data", "Theory", "Question", "Example", "Critique"]
print(f"data:  {df.shape}, {df['unit'].nunique()} units, {len(codes)} codes")

# --- pyena side ---
pyena_avs, unit_ids_py, _ = compute_all_avs(
    df, codes,
    unit_col="unit", conversation_col="conversation",
    window_size=4, binary=True,
)
print(f"pyena AVs: {pyena_avs.shape}")

# --- rENA side ---
# Push the DataFrame to R and run ena.accumulate.data
with (ro.default_converter + ro.pandas2ri.converter).context():
    ro.globalenv["toy_data"] = ro.conversion.get_conversion().py2rpy(df)

ro.r('''
accum <- ena.accumulate.data(
    units        = toy_data[, c("unit", "group")],
    conversation = toy_data[, c("conversation", "group")],
    codes        = toy_data[, c("Data", "Theory", "Question", "Example", "Critique")],
    window.size.back = 4
)
''')

# Pull adjacency vectors back into Python
rena_avs = np.array(ro.r('as.matrix(accum$connection.counts[, -c(1,2,3)])'))
rena_unit_ids = list(ro.r('as.character(accum$connection.counts$ENA_UNIT)'))
print(f"rENA AVs:  {rena_avs.shape}")

# rENA prefixes unit IDs with the group label ("At01::A_tight"); strip it
rena_unit_only = [u.split("::")[0] for u in rena_unit_ids]

# Align pyena rows to rENA's order
py_idx = {u: i for i, u in enumerate(unit_ids_py)}
pyena_aligned = np.array([pyena_avs[py_idx[u]] for u in rena_unit_only])

# Compare
diff = np.abs(pyena_aligned - rena_avs)
print()
print(f"=== Adjacency vectors ===")
print(f"max |Δ|:               {diff.max():.2e}")
print(f"mismatched cells > 0:  {(diff > 0).sum()} / {diff.size}")
print(f"verdict:               {'✓ exact match' if diff.max() == 0 else '✗ mismatch'}")

data:  (1800, 9), 90 units, 5 codes
pyena AVs: (90, 10)
rENA AVs:  (90, 10)

=== Adjacency vectors ===
max |Δ|:               0.00e+00
mismatched cells > 0:  0 / 900
verdict:               ✓ exact match


## Stage 2: SVD coordinates

After sphere normalization and centering, pyena's `svd_project` and rENA's
default `ena.svd` rotation should produce identical 2D coordinates (up to an
arbitrary per-axis sign).

In [8]:
from pyena.core.normalize import sphere_normalize
from pyena.core.projection import svd_project

TOL = 1e-10

# --- pyena side ---
av_normalized = sphere_normalize(pyena_avs)
proj = svd_project(av_normalized, n_components=2)
pyena_coords = proj.coords
print(f"pyena coords: {pyena_coords.shape}")

# --- rENA side ---
ro.r('''
set_svd <- ena.make.set(
    enadata = accum,
    center.align.to.origin = FALSE
)
''')
rena_svd = np.array(ro.r('as.matrix(set_svd$points[, .(SVD1, SVD2)])'))
rena_units = [u.split("::")[0] for u
              in ro.r('as.character(set_svd$points$ENA_UNIT)')]
print(f"rENA coords:  {rena_svd.shape}")

# Align pyena rows to rENA order
py_idx = {u: i for i, u in enumerate(unit_ids_py)}
pyena_aligned_svd = np.array([pyena_coords[py_idx[u]] for u in rena_units])

# Sign-align each axis (SVD signs are arbitrary)
pyena_signed_svd = pyena_aligned_svd.copy()
for k in range(2):
    if np.dot(pyena_signed_svd[:, k], rena_svd[:, k]) < 0:
        pyena_signed_svd[:, k] = -pyena_signed_svd[:, k]

diff_svd = np.abs(pyena_signed_svd - rena_svd)
print()
print(f"=== SVD coordinates (sign-aligned) ===")
print(f"max |Δ|:                       {diff_svd.max():.2e}")
print(f"values exceeding tol={TOL}:    {(diff_svd > TOL).sum()} / {diff_svd.size}")
print(f"verdict:                        {'✓ machine-epsilon match' if diff_svd.max() < TOL else '✗ mismatch'}")
print()
print("First 3 units:")
print(f"{'unit':<6}{'rENA SVD1':>12}{'pyena':>12}{'rENA SVD2':>14}{'pyena':>12}")
for i in range(3):
    print(f"{rena_units[i]:<6}"
          f"{rena_svd[i,0]:+12.6f}{pyena_signed_svd[i,0]:+12.6f}"
          f"{rena_svd[i,1]:+14.6f}{pyena_signed_svd[i,1]:+12.6f}")

pyena coords: (90, 2)
rENA coords:  (90, 2)

=== SVD coordinates (sign-aligned) ===
max |Δ|:                       8.88e-16
values exceeding tol=1e-10:    0 / 180
verdict:                        ✓ machine-epsilon match

First 3 units:
unit     rENA SVD1       pyena     rENA SVD2       pyena
At01     +0.448273   +0.448273     -0.167357   -0.167357
At02     +0.469352   +0.469352     -0.324044   -0.324044
At03     +0.377977   +0.377977     +0.018921   +0.018921


## Stage 3: Means rotation

The final and most algorithmically involved stage is means rotation —
rENA's `ena.rotate.by.mean`. pyena reimplements rENA's `orthogonal_svd`
helper (which uses an explicit QR-based orthogonal complement, not just
post-deflation SVD), giving exact reproduction of rENA's MR1/SVD2 output.

We compare on the `A_tight` vs `B_tight` contrast.

In [9]:
from pyena.core.projection import means_rotation

TOL = 1e-10

# --- pyena side ---
# Reuse the centered matrix from svd_project output for fair comparison
centered = proj.centered

# Get group labels in the same order as unit_ids_py
group_by_unit = dict(zip(df['unit'], df['group']))
group_labels = np.array([group_by_unit[u] for u in unit_ids_py])

mr_result = means_rotation(centered, group_labels,
                            group_a='A_tight', group_b='B_tight')
pyena_mr = mr_result.points[:, :2]   # MR1, SVD2
print(f"pyena MR coords: {pyena_mr.shape}")

# --- rENA side ---
ro.r('''
set_mr <- ena.make.set(
    enadata = accum,
    rotation.by = ena.rotate.by.mean,
    rotation.params = list(
        accum$meta.data$group == "A_tight",
        accum$meta.data$group == "B_tight"
    ),
    center.align.to.origin = FALSE
)
''')
rena_mr = np.array(ro.r('as.matrix(set_mr$points[, .(MR1, SVD2)])'))
rena_units_mr = [u.split("::")[0] for u
                 in ro.r('as.character(set_mr$points$ENA_UNIT)')]
print(f"rENA MR coords:  {rena_mr.shape}")

# Align pyena rows to rENA order
pyena_aligned_mr = np.array([pyena_mr[py_idx[u]] for u in rena_units_mr])

# Sign-align
pyena_signed_mr = pyena_aligned_mr.copy()
for k in range(2):
    if np.dot(pyena_signed_mr[:, k], rena_mr[:, k]) < 0:
        pyena_signed_mr[:, k] = -pyena_signed_mr[:, k]

diff_mr = np.abs(pyena_signed_mr - rena_mr)
print()
print(f"=== Means-rotation coordinates (sign-aligned) ===")
print(f"max |Δ|:                       {diff_mr.max():.2e}")
print(f"values exceeding tol={TOL}:    {(diff_mr > TOL).sum()} / {diff_mr.size}")
print(f"verdict:                        {'✓ machine-epsilon match' if diff_mr.max() < TOL else '✗ mismatch'}")
print()
print("First 3 units:")
print(f"{'unit':<6}{'rENA MR1':>12}{'pyena':>12}{'rENA SVD2':>14}{'pyena':>12}")
for i in range(3):
    print(f"{rena_units_mr[i]:<6}"
          f"{rena_mr[i,0]:+12.6f}{pyena_signed_mr[i,0]:+12.6f}"
          f"{rena_mr[i,1]:+14.6f}{pyena_signed_mr[i,1]:+12.6f}")

pyena MR coords: (90, 2)
rENA MR coords:  (90, 2)

=== Means-rotation coordinates (sign-aligned) ===
max |Δ|:                       5.55e-16
values exceeding tol=1e-10:    0 / 180
verdict:                        ✓ machine-epsilon match

First 3 units:
unit      rENA MR1       pyena     rENA SVD2       pyena
At01     -0.446701   -0.446701     -0.171405   -0.171405
At02     -0.461169   -0.461169     -0.328928   -0.328928
At03     -0.380750   -0.380750     +0.016143   +0.016143


## Summary

Three core stages of pyena have been reproduced against rENA 0.3.1 on the
same 90-unit toy dataset:

| Stage              | Cells / values | Max abs Δ      | Verdict                  |
|--------------------|----------------|----------------|--------------------------|
| Adjacency vectors  | 900            | 0              | exact (integer match)    |
| SVD coordinates    | 180            | 8.88 × 10⁻¹⁶   | machine-epsilon match    |
| Means rotation     | 180            | 5.55 × 10⁻¹⁶   | machine-epsilon match    |

The differences for SVD and MR are at the level of IEEE 754 double-precision
arithmetic and stem from rENA computing through the LAPACK routines exposed
by R, while pyena uses NumPy's LAPACK bindings; the algorithms themselves
match exactly (see `pyena.core.projection.orthogonal_svd` for the
reimplementation of rENA's helper function).

For pyena's pure-Python regression suite (no R required), run `pytest` from
the repository root — 101 tests cover the same regressions plus
mathematical-property checks and external cross-checks against scipy.